# Addresses26 — v5 Agent Tables Builder

This notebook starts building the actual estate-agent attribution tables for the Addresses26 project.

It does **not** scrape portals automatically. It uses your existing verified Land Registry BN sales data and a manual review sheet where agent evidence can be entered.

## Outputs

The notebook creates:

- `manual_review_queue_bn_last5y.xlsx`
- `agent_attribution_links_bn_last5y.csv`
- `agents_canonical_bn_last5y.csv`
- `agent_summary_by_agent_bn_last5y.csv`
- `agent_summary_by_postcode_district_bn_last5y.csv`
- `agent_summary_by_year_bn_last5y.csv`
- `agent_review_progress_bn_last5y.csv`
- `agent_tables_manifest_bn_last5y.json`

## Core idea

Land Registry remains the verified factual layer.  
Agent attribution is stored separately as an evidence-backed enrichment layer.

In [1]:
# 01 — Mount Drive and imports

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re
import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import quote_plus

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

BASE = Path('/content/drive/MyDrive/Addresses26_Data')
DERIVED = BASE / 'derived' / 'land_registry_price_paid'
REPORTS = BASE / 'reports' / 'agent_attribution'
REPORTS.mkdir(parents=True, exist_ok=True)

POSTCODE_PREFIX = 'BN'
YEARS = list(range(2021, 2026))  # last 5 completed/near-current years in your current pilot
SAMPLE_ROWS_PER_YEAR = 50
RANDOM_SEED = 26

print('BASE   :', BASE)
print('DERIVED:', DERIVED)
print('REPORTS:', REPORTS)
print('YEARS  :', YEARS)

Mounted at /content/drive
BASE   : /content/drive/MyDrive/Addresses26_Data
DERIVED: /content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid
REPORTS: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution
YEARS  : [2021, 2022, 2023, 2024, 2025]


In [2]:
# 02 — Helper functions

def normalise_text(x):
    if pd.isna(x):
        return ''
    x = str(x).strip()
    x = re.sub(r'\s+', ' ', x)
    return x

def normalise_postcode(x):
    x = normalise_text(x).upper()
    x = re.sub(r'\s+', '', x)
    if len(x) > 3:
        return x[:-3] + ' ' + x[-3:]
    return x

def postcode_district_from_postcode(x):
    pc = normalise_postcode(x)
    if not pc:
        return ''
    outward = pc.split(' ')[0]
    return outward

def clean_agent_name(name):
    name = normalise_text(name)
    if not name:
        return ''
    # remove common legal suffix noise while preserving recognisable branch names
    name = re.sub(r'\b(LTD|LIMITED|LLP|PLC)\.?\b', '', name, flags=re.I)
    name = re.sub(r'\s+', ' ', name).strip(' ,.-')
    # title case is useful for display, but keep common all-caps acronyms reasonably readable
    return name.title()

def canonical_agent_key(name):
    name = clean_agent_name(name).lower()
    name = re.sub(r'[^a-z0-9]+', ' ', name)
    name = re.sub(r'\s+', ' ', name).strip()
    return name

def stable_id(prefix, parts):
    joined = '|'.join([normalise_text(p) for p in parts])
    h = hashlib.sha1(joined.encode('utf-8')).hexdigest()[:16]
    return f'{prefix}_{h}'

def money_to_numeric(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float, np.integer, np.floating)):
        return x
    s = str(x).replace('£','').replace(',','').strip()
    return pd.to_numeric(s, errors='coerce')

def make_search_query(row):
    address_bits = [
        row.get('paon', ''),
        row.get('saon', ''),
        row.get('street', ''),
        row.get('locality', ''),
        row.get('town_city', ''),
        row.get('postcode', ''),
    ]
    address = ' '.join([normalise_text(v) for v in address_bits if normalise_text(v)])
    price = row.get('price', '')
    year = row.get('sale_year', '')
    return f'"{address}" "{price}" estate agent sold {year}'

def make_google_url(query):
    return 'https://www.google.com/search?q=' + quote_plus(query)

def make_rightmove_house_prices_url(postcode):
    pc = quote_plus(normalise_postcode(postcode))
    return f'https://www.rightmove.co.uk/house-prices/{pc}.html'

def make_zoopla_url(postcode):
    pc = quote_plus(normalise_postcode(postcode))
    return f'https://www.zoopla.co.uk/house-prices/{pc}/'

def make_otm_url(query):
    return 'https://www.onthemarket.com/for-sale/property/?search-term=' + quote_plus(query)

In [3]:
# 03 — Load verified BN Land Registry parquet files

frames = []
missing_files = []

for year in YEARS:
    path = DERIVED / f'pp_bn_standard_{year}.parquet'
    if path.exists():
        df = pd.read_parquet(path)
        df['source_year_file'] = year
        frames.append(df)
        print('Loaded', path.name, len(df))
    else:
        missing_files.append(str(path))
        print('MISSING:', path)

assert frames, 'No parquet files found. Check DERIVED path and year files.'

sales = pd.concat(frames, ignore_index=True)

# Standardise common fields.
sales.columns = [str(c).strip().lower() for c in sales.columns]

required_any = ['price', 'date_of_transfer', 'postcode']
missing = [c for c in required_any if c not in sales.columns]
assert not missing, f'Missing required fields in parquet data: {missing}'

sales['price'] = sales['price'].apply(money_to_numeric)
sales['date_of_transfer'] = pd.to_datetime(sales['date_of_transfer'], errors='coerce')
sales['sale_year'] = sales['date_of_transfer'].dt.year
sales['postcode'] = sales['postcode'].apply(normalise_postcode)
sales['postcode_district'] = sales['postcode'].apply(postcode_district_from_postcode)

# Ensure BN-only standard data as a safety check.
sales = sales[sales['postcode'].str.startswith(POSTCODE_PREFIX, na=False)].copy()

if 'ppd_category_type' in sales.columns:
    sales = sales[sales['ppd_category_type'].astype(str).str.upper().eq('A')].copy()

# Build stable transaction ids from factual fields.
id_cols = [c for c in ['price','date_of_transfer','postcode','paon','saon','street','town_city'] if c in sales.columns]
sales['transaction_id'] = sales.apply(lambda r: stable_id('txn', [r.get(c, '') for c in id_cols]), axis=1)

print('Sales rows:', len(sales))
display(sales[['transaction_id','date_of_transfer','sale_year','price','postcode','postcode_district']].head())

Loaded pp_bn_standard_2021.parquet 18279
Loaded pp_bn_standard_2022.parquet 14934
Loaded pp_bn_standard_2023.parquet 11515
Loaded pp_bn_standard_2024.parquet 12532
Loaded pp_bn_standard_2025.parquet 11425
Sales rows: 68685


,transaction_id,date_of_transfer,sale_year,price,postcode,postcode_district
0,txn_5506fe91ce76af11,2021-01-03,2021,650000,BN3 5DL,BN3
1,txn_f481d5dfacc2afa4,2021-01-04,2021,265000,BN1 3RT,BN1
2,txn_e6112bec9bd456b4,2021-01-04,2021,505000,BN16 2EA,BN16
3,txn_92292546ad240a20,2021-01-04,2021,430000,BN2 5AD,BN2
4,txn_e606e965a6d49bad,2021-01-04,2021,800000,BN20 8DL,BN20


In [4]:
# 04 — Create or load manual review queue

QUEUE_CSV = REPORTS / 'manual_review_queue_bn_last5y.csv'
QUEUE_XLSX = REPORTS / 'manual_review_queue_bn_last5y.xlsx'

if QUEUE_CSV.exists():
    queue = pd.read_csv(QUEUE_CSV)
    print('Loaded existing queue:', QUEUE_CSV, len(queue))
else:
    # Balanced sample by year, capped by available rows.
    sampled = []
    rng = np.random.default_rng(RANDOM_SEED)
    for year, g in sales.groupby('sale_year'):
        if year not in YEARS:
            continue
        n = min(SAMPLE_ROWS_PER_YEAR, len(g))
        sampled.append(g.sample(n=n, random_state=RANDOM_SEED + int(year)))
    queue = pd.concat(sampled, ignore_index=True) if sampled else sales.sample(n=min(250, len(sales)), random_state=RANDOM_SEED).copy()

    # Add review/search columns.
    queue['address_search_query'] = queue.apply(make_search_query, axis=1)
    queue['google_search_url'] = queue['address_search_query'].apply(make_google_url)
    queue['rightmove_house_prices_url'] = queue['postcode'].apply(make_rightmove_house_prices_url)
    queue['zoopla_house_prices_url'] = queue['postcode'].apply(make_zoopla_url)
    queue['onthemarket_search_url'] = queue['address_search_query'].apply(make_otm_url)

    queue['manual_agent_name'] = ''
    queue['source_url'] = ''
    queue['evidence_type'] = ''
    queue['confidence'] = ''
    queue['review_status'] = 'pending'
    queue['review_notes'] = ''
    queue['reviewed_by'] = ''
    queue['reviewed_utc'] = ''

    queue.to_csv(QUEUE_CSV, index=False)
    print('Created new queue:', QUEUE_CSV, len(queue))

# Ensure expected manual columns exist even if old queue is loaded.
manual_cols = {
    'manual_agent_name': '',
    'source_url': '',
    'evidence_type': '',
    'confidence': '',
    'review_status': 'pending',
    'review_notes': '',
    'reviewed_by': '',
    'reviewed_utc': '',
}
for c, default in manual_cols.items():
    if c not in queue.columns:
        queue[c] = default

# Ensure useful URL columns exist.
if 'address_search_query' not in queue.columns:
    queue['address_search_query'] = queue.apply(make_search_query, axis=1)
if 'google_search_url' not in queue.columns:
    queue['google_search_url'] = queue['address_search_query'].apply(make_google_url)
if 'rightmove_house_prices_url' not in queue.columns:
    queue['rightmove_house_prices_url'] = queue['postcode'].apply(make_rightmove_house_prices_url)
if 'zoopla_house_prices_url' not in queue.columns:
    queue['zoopla_house_prices_url'] = queue['postcode'].apply(make_zoopla_url)
if 'onthemarket_search_url' not in queue.columns:
    queue['onthemarket_search_url'] = queue['address_search_query'].apply(make_otm_url)

# Save Excel version for easier manual editing.
try:
    with pd.ExcelWriter(QUEUE_XLSX, engine='openpyxl') as writer:
        queue.to_excel(writer, sheet_name='manual_review_queue', index=False)
    print('Saved Excel review queue:', QUEUE_XLSX)
except Exception as e:
    print('Could not save Excel queue, CSV is still available:', e)

print('Review status:')
display(queue.groupby(['review_status','confidence'], dropna=False).size().reset_index(name='rows'))
print('Sample by year:')
display(queue.groupby('sale_year').size().reset_index(name='sample_rows'))

Loaded existing queue: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/manual_review_queue_bn_last5y.csv 250
Saved Excel review queue: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/manual_review_queue_bn_last5y.xlsx
Review status:


,review_status,confidence,rows
0,pending,NaN,250


Sample by year:


,sale_year,sample_rows
0,2021,50
1,2022,50
2,2023,50
3,2024,50
4,2025,50


## Manual review instructions

Open the Excel or CSV queue in Google Drive:

`reports/agent_attribution/manual_review_queue_bn_last5y.xlsx`

For a small first run, review 25–50 rows only.

Fill:

- `manual_agent_name`
- `source_url`
- `evidence_type`
- `confidence`
- `review_status`
- `review_notes`

Suggested values:

- `review_status`: `confirmed`, `candidate`, `unknown`, `rejected`, `pending`
- `confidence`: `high`, `medium`, `low`, blank
- `evidence_type`: `portal_listing`, `portal_sold_page`, `agent_page`, `google_result`, `manual_other`

Then save the file and rerun from the next cell onward.

In [5]:
# 05 — Reload reviewed queue if edited

# Preference: Excel if it exists and has been manually edited; otherwise CSV.
if QUEUE_XLSX.exists():
    reviewed = pd.read_excel(QUEUE_XLSX, sheet_name='manual_review_queue')
    print('Reloaded reviewed Excel queue:', QUEUE_XLSX, len(reviewed))
else:
    reviewed = pd.read_csv(QUEUE_CSV)
    print('Reloaded reviewed CSV queue:', QUEUE_CSV, len(reviewed))

reviewed.columns = [str(c).strip() for c in reviewed.columns]

# Normalise review fields.
for c in manual_cols:
    if c not in reviewed.columns:
        reviewed[c] = manual_cols[c]
    reviewed[c] = reviewed[c].fillna('').astype(str).str.strip()

reviewed['manual_agent_clean'] = reviewed['manual_agent_name'].apply(clean_agent_name)
reviewed['agent_key'] = reviewed['manual_agent_clean'].apply(canonical_agent_key)

reviewed['review_status'] = reviewed['review_status'].str.lower().replace({
    '': 'pending',
    'confirm': 'confirmed',
    'confirmed ': 'confirmed',
    'maybe': 'candidate',
    'possible': 'candidate',
    'not found': 'unknown',
    'no result': 'unknown',
})

reviewed['confidence'] = reviewed['confidence'].str.lower().replace({
    '': '',
    'hi': 'high',
    'med': 'medium',
    'mid': 'medium',
})

valid_status = {'pending','confirmed','candidate','unknown','rejected'}
valid_conf = {'','high','medium','low'}

reviewed['status_valid'] = reviewed['review_status'].isin(valid_status)
reviewed['confidence_valid'] = reviewed['confidence'].isin(valid_conf)

bad = reviewed[~reviewed['status_valid'] | ~reviewed['confidence_valid']]
if len(bad):
    print('Rows with invalid review_status/confidence:')
    display(bad[['transaction_id','review_status','confidence','manual_agent_name','source_url']].head(20))
else:
    print('All review status/confidence values valid.')

display(reviewed.groupby(['review_status','confidence'], dropna=False).size().reset_index(name='rows'))

Reloaded reviewed Excel queue: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/manual_review_queue_bn_last5y.xlsx 250
All review status/confidence values valid.


,review_status,confidence,rows
0,pending,,250


In [6]:
# 06 — Build transaction-agent attribution link table

usable_status = {'confirmed', 'candidate'}
links = reviewed[
    reviewed['review_status'].isin(usable_status) &
    reviewed['manual_agent_clean'].ne('') &
    reviewed['source_url'].ne('')
].copy()

links['agent_id'] = links['agent_key'].apply(lambda x: stable_id('agent', [x]))
links['attribution_id'] = links.apply(lambda r: stable_id('attr', [r.get('transaction_id',''), r.get('agent_id',''), r.get('source_url','')]), axis=1)
links['created_utc'] = datetime.now(timezone.utc).isoformat()

# Keep useful columns only, but tolerate missing source columns.
link_cols = [
    'attribution_id','transaction_id','agent_id','manual_agent_clean','agent_key',
    'review_status','confidence','evidence_type','source_url','review_notes',
    'date_of_transfer','sale_year','price','postcode','postcode_district',
    'property_type','new_build_flag','tenure_type',
    'paon','saon','street','locality','town_city','district','county',
    'created_utc'
]
link_cols = [c for c in link_cols if c in links.columns]
links_out = links[link_cols].copy()

LINKS_CSV = REPORTS / 'agent_attribution_links_bn_last5y.csv'
links_out.to_csv(LINKS_CSV, index=False)

print('Attribution links created:', len(links_out))
print('Saved:', LINKS_CSV)
display(links_out.head(20))

Attribution links created: 0
Saved: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_attribution_links_bn_last5y.csv


,attribution_id,agent_id,manual_agent_clean,agent_key,review_status,confidence,evidence_type,source_url,review_notes,date_of_transfer,sale_year,price,postcode,postcode_district,property_type,created_utc


In [7]:
# 07 — Build canonical agents table

if len(links_out):
    agents = (
        links_out
        .groupby(['agent_id','agent_key','manual_agent_clean'], dropna=False)
        .agg(
            attributed_transactions=('transaction_id','nunique'),
            first_sale_date=('date_of_transfer','min'),
            last_sale_date=('date_of_transfer','max'),
            postcode_districts=('postcode_district', lambda s: ', '.join(sorted(set([str(x) for x in s if str(x) != 'nan'])))),
            high_confidence_rows=('confidence', lambda s: (s == 'high').sum()),
            medium_confidence_rows=('confidence', lambda s: (s == 'medium').sum()),
            low_confidence_rows=('confidence', lambda s: (s == 'low').sum()),
        )
        .reset_index()
        .rename(columns={'manual_agent_clean':'agent_display_name'})
        .sort_values(['attributed_transactions','agent_display_name'], ascending=[False, True])
    )
else:
    agents = pd.DataFrame(columns=[
        'agent_id','agent_key','agent_display_name','attributed_transactions',
        'first_sale_date','last_sale_date','postcode_districts',
        'high_confidence_rows','medium_confidence_rows','low_confidence_rows'
    ])

AGENTS_CSV = REPORTS / 'agents_canonical_bn_last5y.csv'
agents.to_csv(AGENTS_CSV, index=False)

print('Canonical agents:', len(agents))
print('Saved:', AGENTS_CSV)
display(agents.head(30))

Canonical agents: 0
Saved: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agents_canonical_bn_last5y.csv


,agent_id,agent_key,agent_display_name,attributed_transactions,first_sale_date,last_sale_date,postcode_districts,high_confidence_rows,medium_confidence_rows,low_confidence_rows


In [8]:
# 08 — Build progress and summary tables

progress = (
    reviewed
    .groupby(['review_status','confidence'], dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values(['review_status','confidence'])
)

progress['pct_of_queue'] = (progress['rows'] / len(reviewed) * 100).round(2) if len(reviewed) else 0

summary_by_year = (
    reviewed
    .groupby(['sale_year','review_status'], dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values(['sale_year','review_status'])
)

summary_by_district = (
    reviewed
    .groupby(['postcode_district','review_status'], dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values(['postcode_district','review_status'])
)

if len(links_out):
    summary_by_agent = (
        links_out
        .groupby(['agent_id','manual_agent_clean'], dropna=False)
        .agg(
            rows=('transaction_id','count'),
            unique_transactions=('transaction_id','nunique'),
            high_confidence=('confidence', lambda s: (s == 'high').sum()),
            medium_confidence=('confidence', lambda s: (s == 'medium').sum()),
            low_confidence=('confidence', lambda s: (s == 'low').sum()),
            postcode_districts=('postcode_district', lambda s: ', '.join(sorted(set([str(x) for x in s if str(x) != 'nan'])))),
        )
        .reset_index()
        .sort_values(['unique_transactions','manual_agent_clean'], ascending=[False, True])
    )
else:
    summary_by_agent = pd.DataFrame(columns=['agent_id','manual_agent_clean','rows','unique_transactions','high_confidence','medium_confidence','low_confidence','postcode_districts'])

PROGRESS_CSV = REPORTS / 'agent_review_progress_bn_last5y.csv'
YEAR_CSV = REPORTS / 'agent_summary_by_year_bn_last5y.csv'
DISTRICT_CSV = REPORTS / 'agent_summary_by_postcode_district_bn_last5y.csv'
AGENT_SUMMARY_CSV = REPORTS / 'agent_summary_by_agent_bn_last5y.csv'

progress.to_csv(PROGRESS_CSV, index=False)
summary_by_year.to_csv(YEAR_CSV, index=False)
summary_by_district.to_csv(DISTRICT_CSV, index=False)
summary_by_agent.to_csv(AGENT_SUMMARY_CSV, index=False)

print('Saved summaries:')
for p in [PROGRESS_CSV, YEAR_CSV, DISTRICT_CSV, AGENT_SUMMARY_CSV]:
    print(' -', p)

print('\nReview progress:')
display(progress)

print('\nTop agents so far:')
display(summary_by_agent.head(20))

Saved summaries:
 - /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_review_progress_bn_last5y.csv
 - /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_summary_by_year_bn_last5y.csv
 - /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_summary_by_postcode_district_bn_last5y.csv
 - /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_summary_by_agent_bn_last5y.csv

Review progress:


,review_status,confidence,rows,pct_of_queue
0,pending,,250,100.0



Top agents so far:


,agent_id,manual_agent_clean,rows,unique_transactions,high_confidence,medium_confidence,low_confidence,postcode_districts


In [9]:
# 09 — Quality checks for the agent tables

checks = {}

checks['queue_rows'] = int(len(reviewed))
checks['usable_links'] = int(len(links_out))
checks['canonical_agents'] = int(len(agents))
checks['pending_rows'] = int((reviewed['review_status'] == 'pending').sum())
checks['confirmed_rows'] = int((reviewed['review_status'] == 'confirmed').sum())
checks['candidate_rows'] = int((reviewed['review_status'] == 'candidate').sum())
checks['unknown_rows'] = int((reviewed['review_status'] == 'unknown').sum())
checks['rejected_rows'] = int((reviewed['review_status'] == 'rejected').sum())

# Required evidence rule: confirmed/candidate rows used for links must have agent and source URL.
if len(links_out):
    checks['links_missing_source_url'] = int((links_out['source_url'].astype(str).str.strip() == '').sum())
    checks['links_missing_agent'] = int((links_out['manual_agent_clean'].astype(str).str.strip() == '').sum())
    checks['duplicate_attribution_ids'] = int(links_out['attribution_id'].duplicated().sum())
else:
    checks['links_missing_source_url'] = 0
    checks['links_missing_agent'] = 0
    checks['duplicate_attribution_ids'] = 0

checks['validation_pass'] = (
    checks['links_missing_source_url'] == 0 and
    checks['links_missing_agent'] == 0 and
    checks['duplicate_attribution_ids'] == 0
)

print(json.dumps(checks, indent=2))

{
  "queue_rows": 250,
  "usable_links": 0,
  "canonical_agents": 0,
  "pending_rows": 250,
  "confirmed_rows": 0,
  "candidate_rows": 0,
  "unknown_rows": 0,
  "rejected_rows": 0,
  "links_missing_source_url": 0,
  "links_missing_agent": 0,
  "duplicate_attribution_ids": 0,
  "validation_pass": true
}


In [10]:
# 10 — Manifest

manifest = {
    'project': 'Addresses26',
    'notebook': 'Addresses26_04_agent_tables_v5_builder.ipynb',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'postcode_prefix': POSTCODE_PREFIX,
    'years': YEARS,
    'sample_rows_per_year': SAMPLE_ROWS_PER_YEAR,
    'random_seed': RANDOM_SEED,
    'inputs': {
        'derived_dir': str(DERIVED),
        'queue_csv': str(QUEUE_CSV),
        'queue_xlsx': str(QUEUE_XLSX),
    },
    'outputs': {
        'links_csv': str(LINKS_CSV),
        'agents_csv': str(AGENTS_CSV),
        'progress_csv': str(PROGRESS_CSV),
        'summary_by_year_csv': str(YEAR_CSV),
        'summary_by_postcode_district_csv': str(DISTRICT_CSV),
        'summary_by_agent_csv': str(AGENT_SUMMARY_CSV),
    },
    'checks': checks,
    'important_note': 'Agent attribution is evidence-backed enrichment, not Land Registry truth. Keep separate from verified transaction core.'
}

MANIFEST_JSON = REPORTS / 'agent_tables_manifest_bn_last5y.json'
MANIFEST_JSON.write_text(json.dumps(manifest, indent=2), encoding='utf-8')

print('Manifest saved:', MANIFEST_JSON)
print(json.dumps(manifest, indent=2)[:2000])

Manifest saved: /content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_tables_manifest_bn_last5y.json
{
  "project": "Addresses26",
  "notebook": "Addresses26_04_agent_tables_v5_builder.ipynb",
  "created_utc": "2026-04-27T14:01:50.924220+00:00",
  "postcode_prefix": "BN",
  "years": [
    2021,
    2022,
    2023,
    2024,
    2025
  ],
  "sample_rows_per_year": 50,
  "random_seed": 26,
  "inputs": {
    "derived_dir": "/content/drive/MyDrive/Addresses26_Data/derived/land_registry_price_paid",
    "queue_csv": "/content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/manual_review_queue_bn_last5y.csv",
    "queue_xlsx": "/content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/manual_review_queue_bn_last5y.xlsx"
  },
  "outputs": {
    "links_csv": "/content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agent_attribution_links_bn_last5y.csv",
    "agents_csv": "/content/drive/MyDrive/Addresses26_Data/reports/agent_attribution/agents_canon

## Recommended next step

1. Open the generated Excel review queue.
2. Manually review 25–50 rows.
3. Save the Excel file.
4. Rerun cells 05–10.
5. Inspect:
   - `agents_canonical_bn_last5y.csv`
   - `agent_attribution_links_bn_last5y.csv`
   - `agent_review_progress_bn_last5y.csv`

If this yields useful matches, the next notebook should add a lightweight dashboard and optional district-specific sampling.